In [ ]:
from demoparser2 import DemoParser
import pandas as pd
from pathlib import Path
import numpy as np

In [ ]:
#BASE_DIR = Path(__file__).resolve().parent
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / 'data'

files = [str(p) for p in list(DATA_DIR.rglob('*.dem'))]

In [2]:
PLAYER_PROPS = [
    "team_name",
    "tick",
    "total_rounds_played",
    "player_name",
    "team_num",
    "health",
    "X", "Y", "Z",
    "velocity_X", "velocity_Y", "velocity_Z",
    "yaw", "pitch",
    "is_match_started",
    "is_alive",
    "active_weapon_name",
    "inventory",
    "has"
    "current_equip_value",
    "team_rounds_total",
    "is_match_started"
]

KEEP_COLS = [
    'tick',
    'steamid',
    'name',
    'tick',
    'round_num',
    'health',
    'X', 'Y', 'Z',
    'velocity_X', 'velocity_Y', 'velocity_Z',
    'yaw', 'pitch',
    'team_name',
    'inventory',
    'is_alive',
    'is_planted',
    'bomb_site',
]

EVENT_PROPS = [
    'round_announce_match_start',
    'cs_round_final_beep',
    'cs_win_panel_match',
    'bomb_planted',
    'bomb_exploded',
    'bomb_defused',
    'grenade_thrown',
    'player_death',
    'player_hurt',
    'round_start',
    'round_end',
]

DEATH_COLS = [
    'tick',
    'user_steamid',
    'dmg_health', 
    'weapon',
    'attacker_steamid', 
    'attackerblind', 
    'attackerinair', 
    'headshot', 
    'hitgroup', 
    'noscope', 
    'penetrated', 
    'thrusmoke', 
    'assistedflash', 
    'assister_steamid',
]

HURT_COLS = [
    'tick',
    'user_steamid',
    'health',
    'dmg_health',
    'weapon',
    'attacker_steamid',
    'hitgroup',
]

NADE_COLS = [
    'tick',
    'user_steamid',
    'weapon',
]

def parse_demo(path):
    parser = DemoParser(path)

    players = parser.parse_ticks(PLAYER_PROPS)
    events = parser.parse_events(EVENT_PROPS, player=["last_place_name"])
    info = parser.parse_header()

    return players, events, info

def parse_file(file):
    players, events, info = parse_demo(file)
    players['steamid'] = players['steamid'].astype(str)
    players = players[players['is_match_started'] == True]
    players = players.copy()
    map_name = info.get('map_name')

    #EVENTS INTO DICT
    frames = {frame[0]:frame[1] for frame in events}
    
    #EXTRACT EVENTS
    match_start = frames.get('round_announce_match_start')
    if match_start is not None:
        match_start = match_start.loc[0, 'tick']
    round_starts = frames.get('cs_round_final_beep')
    match_end = frames.get('cs_win_panel_match')
    if match_end is not None:
        match_end = match_end.loc[0, 'tick']
    rounds = frames.get('round_start')
    if rounds is not None:
        rounds = rounds.loc[rounds.groupby('round')['tick'].idxmax()]
    round_ends = frames.get('round_end')
    if round_ends is not None:
        round_ends = round_ends[(round_ends['reason'].notna()) & (round_ends['tick'] != 0) & (round_ends['winner'].notna())]
        round_ends['round'] = np.arange(1, len(round_ends) + 1)
    deaths = frames.get('player_death')
    if deaths is not None:
        deaths = deaths[DEATH_COLS]
    hurt = frames.get('player_hurt')
    if hurt is not None:
        hurt = hurt[HURT_COLS]
    nades = frames.get('grenade_thrown')
    if nades is not None:
        nades = nades[NADE_COLS]
    planted = frames.get('bomb_planted')
    defused = frames.get('bomb_defused')
    explode = frames.get('bomb_exploded')

    #START AND END OF ROUND
    round_numbers = (players[players['tick'].isin(round_starts['tick'])][['tick', 'total_rounds_played']]
                        .rename(columns={'tick':'start_tick', 'total_rounds_played':'start_total_rounds_played'}))
    
    #MATCH BOUNDARIES
    if match_end:
        players = players[players['tick'] <= match_end]
    if match_start:
        players = players[players['tick'] >= match_start]
    players = players.copy()

    #START TICK MERGE
    players = pd.merge_asof(players, round_numbers, left_on='tick', right_on='start_tick', direction='backward')
    in_round = players['total_rounds_played'] == players['start_total_rounds_played']
    shifted_in_round = in_round.shift(1).fillna(False)
    players = players[in_round | shifted_in_round]
    players['round_num'] = players['total_rounds_played'] + 1

    #BOMB INFO
    if planted is not None:
        bomb_df = players.merge(planted.drop(columns='user_name'), left_on=['tick', 'steamid'], right_on=['tick', 'user_steamid'], how='left').rename(columns={'user_last_place_name':'bomb_site'})
        players['bomb_planted'] = bomb_df['site'].notna().astype(int)

        players['is_planted'] = np.where(players['bomb_planted']==1, True, pd.NA)
        players['is_planted'] = players.groupby('round_num')['is_planted'].ffill()
        players['is_planted'] = players['is_planted'].astype('boolean').fillna(False)

        players['bomb_site'] = np.where(bomb_df['bomb_site'].notna(), bomb_df['bomb_site'], pd.NA)
        players['bomb_site'] = players.groupby('round_num')['bomb_site'].ffill()

    #GET TICKS WHERE EVENT OCCURS
    important_ticks = pd.concat([
        round_starts['tick'],
        round_ends['tick'],
        deaths['tick'],
        hurt['tick'],
        nades['tick'],
        planted['tick'],
        defused['tick'],
        explode['tick']
    ])

    #CREATE MASKS FOR EVENTS
    important_ticks = important_ticks[important_ticks.isin(players['tick'])]
    event_mask = players['tick'].isin(important_ticks)

    #CREATE MASKS FOR 8TH TICKS
    players['round_tick'] = players['tick']-players['start_tick']
    eighth_mask = (players['round_tick'] % 8 == 0)

    players = players[event_mask | eighth_mask]
    players = players[KEEP_COLS]

    return {
        'map_name': map_name,
        'ticks': players,
        'rounds': rounds,
        'ends': round_ends,
        'deaths': deaths,
        'hurt': hurt,
        'nades': nades,
        'plant': planted,
        'defuse': defused,
        'explode': explode,
    }

In [3]:
players = parse_file(files[0])

NameError: name 'files' is not defined

In [27]:
players[(players['steamid']=='76561197989430253') & (players['total_rounds_played']==1)][['X','Y','Z','tick','team_rounds_total']]

,X,Y,Z,tick,team_rounds_total
98090,-1929.000000,-1025.000000,-415.968750,11281,1.0
98100,-1928.897095,-1025.124023,-415.968750,11282,1.0
98110,-1928.620850,-1025.456787,-415.968750,11283,1.0
98120,-1928.203491,-1025.959473,-415.968750,11284,1.0
98130,-1927.645020,-1026.632080,-415.968750,11285,1.0
...,...,...,...,...,...
191420,652.852112,414.154846,-536.336609,20614,1.0
191430,652.852112,414.154846,-536.336609,20615,1.0
191440,652.852112,414.154846,-536.336609,20616,1.0
191450,652.852112,414.154846,-536.336609,20617,1.0


In [25]:
players[players['is_planted'] == True]

,inventory,total_rounds_played,is_match_started,player_name,health,team_num,current_equip_value,team_rounds_total,team_name,team_clan_name,...,active_weapon_name,tick,steamid,name,start_tick,start_total_rounds_played,round_num,bomb_planted,is_planted,bomb_site
70619,"[Butterfly Knife, P250]",0,True,broky,100.0,2.0,950.0,0.0,TERRORIST,FaZe Clan,...,P250,8532,76561198201620490,broky,1471,0,1,1,True,BombsiteB
70620,[],0,True,karrigan,0.0,2.0,700.0,0.0,TERRORIST,FaZe Clan,...,None,8533,76561197989430253,karrigan,1471,0,1,0,True,BombsiteB
70621,[],0,True,ZywOo,0.0,3.0,950.0,0.0,CT,Team Vitality,...,None,8533,76561198113666193,ZywOo,1471,0,1,0,True,BombsiteB
70622,"[M9 Bayonet, USP-S]",0,True,apEX,24.0,3.0,700.0,0.0,CT,Team Vitality,...,USP-S,8533,76561197989744167,apEX,1471,0,1,0,True,BombsiteB
70623,[],0,True,ropz,0.0,3.0,850.0,0.0,CT,Team Vitality,...,None,8533,76561197991272318,ropz,1471,0,1,0,True,BombsiteB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1582995,[],16,True,jcobbb,0.0,3.0,5800.0,11.0,CT,FaZe Clan,...,None,159806,76561198178737429,jcobbb,151477,16,17,0,True,BombsiteB
1582996,[],16,True,mezii,0.0,2.0,4100.0,5.0,TERRORIST,Team Vitality,...,None,159806,76561197973140692,mezii,151477,16,17,0,True,BombsiteB
1582997,"[Butterfly Knife, USP-S, AWP, High Explosive G...",16,True,Twistzz,94.0,3.0,6650.0,11.0,CT,FaZe Clan,...,Butterfly Knife,159806,76561198016255205,Twistzz,151477,16,17,0,True,BombsiteB
1582998,[],16,True,broky,0.0,3.0,7250.0,11.0,CT,FaZe Clan,...,None,159806,76561198201620490,broky,151477,16,17,0,True,BombsiteB


In [45]:
players

,inventory,total_rounds_played,is_match_started,player_name,health,team_num,current_equip_value,team_rounds_total,team_name,team_clan_name,...,active_weapon_name,tick,steamid,name,start_tick,start_total_rounds_played,round_num,bomb_planted,is_planted,bomb_site
0,"[Skeleton Knife, Glock-18]",0,True,slaxz-,100,2,850,0,TERRORIST,M80,...,Skeleton Knife,1358,76561198064353169,slaxz-,1358,0,1,0.0,False,NaN
1,"[Butterfly Knife, Glock-18, Smoke Grenade, Fla...",0,True,s1n,100,2,900,0,TERRORIST,M80,...,Butterfly Knife,1358,76561198190463461,s1n,1358,0,1,0.0,False,NaN
2,"[M9 Bayonet, Glock-18]",0,True,JBa,100,2,850,0,TERRORIST,M80,...,M9 Bayonet,1358,76561198184153575,JBa,1358,0,1,0.0,False,NaN
3,"[Skeleton Knife, USP-S]",0,True,REZ,100,3,850,0,CT,GamerLegion,...,USP-S,1358,76561198034172415,REZ,1358,0,1,0.0,False,NaN
4,"[M9 Bayonet, USP-S]",0,True,hypex,100,3,850,0,CT,GamerLegion,...,M9 Bayonet,1358,76561198254686734,hypex,1358,0,1,0.0,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2124165,[],21,True,ztr,0,2,5100,12,TERRORIST,GamerLegion,...,None,213801,76561199009468655,ztr,210936,21,22,NaN,False,BombsiteA
2124166,"[Karambit, Glock-18, AK-47, High Explosive Gre...",21,True,Tauson,72,2,5100,12,TERRORIST,GamerLegion,...,AK-47,213801,76561198316706174,Tauson,210936,21,22,NaN,False,BombsiteA
2124167,[],21,True,Lake,0,3,4150,9,CT,M80,...,None,213801,76561198337484196,Lake,210936,21,22,NaN,False,BombsiteA
2124168,"[Skeleton Knife, Glock-18, AK-47]",21,True,Snax,100,2,5100,12,TERRORIST,GamerLegion,...,AK-47,213801,76561197982141573,Snax,210936,21,22,NaN,False,BombsiteA


In [ ]:
players = players.drop(columns=['is_match_started', ''])


In [ ]:
players[['']]

0          TERRORIST
1                 CT
2                 CT
3                 CT
4                 CT
             ...    
1707635           CT
1707636    TERRORIST
1707637           CT
1707638           CT
1707639           CT
Name: team_name, Length: 1205460, dtype: object

In [ ]:
KEEP_COLS = [
    'tick',
    'steamid',
    'name',
    'tick',
    'round_num',
    'health',
    'X', 'Y', 'Z',
    'velocity_X', 'velocity_Y',
    'yaw', 'pitch',
    'team_name',
    'inventory',
    'is_alive',
    'is_planted',
    'bomb_site',
]

In [43]:
files

['/shared_folder/cs2_project/data/demo2/gamerlegion-vs-m80-m1-mirage.dem',
 '/shared_folder/cs2_project/data/demo2/gamerlegion-vs-m80-m2-inferno.dem',
 '/shared_folder/cs2_project/data/demo1/vitality-vs-faze-m1-nuke.dem',
 '/shared_folder/cs2_project/data/demo1/vitality-vs-faze-m4-overpass.dem',
 '/shared_folder/cs2_project/data/demo1/vitality-vs-faze-m3-inferno.dem',
 '/shared_folder/cs2_project/data/demo1/vitality-vs-faze-m2-dust2.dem']

In [46]:
players, events, info = parse_demo(files[0])

In [48]:
test = {k[0]: k[1] for k in events}

In [50]:
test['round_end']

,reason,round,tick,winner
0,ct_killed,2,6309,T
1,ct_killed,3,16196,T
2,bomb_exploded,4,26269,T
3,bomb_defused,5,35431,CT
4,t_killed,6,43157,CT
5,t_killed,7,47527,CT
6,t_killed,8,54863,CT
7,t_killed,9,58856,CT
8,ct_killed,10,68182,T
9,bomb_defused,11,79280,CT


In [54]:
players[players['tick'] == 189630]

,inventory,total_rounds_played,is_match_started,player_name,health,team_num,current_equip_value,team_rounds_total,team_name,team_clan_name,...,active_weapon_name,tick,steamid,name,start_tick,start_total_rounds_played,round_num,bomb_planted,is_planted,bomb_site


In [55]:
test['player_death']

,assistedflash,assister_last_place_name,assister_name,assister_steamid,attacker_last_place_name,attacker_name,attacker_steamid,attackerblind,attackerinair,distance,...,thrusmoke,tick,user_last_place_name,user_name,user_steamid,weapon,weapon_fauxitemid,weapon_itemid,weapon_originalowner_xuid,wipe
0,False,None,None,None,SideAlley,Lake,76561198337484196,False,False,26.789347,...,False,3561,TopofMid,ztr,76561199009468655,glock,17293822569183969284,35627379237,,0
1,True,BombsiteA,s1n,76561198190463461,CTSpawn,JBa,76561198184153575,False,False,11.903322,...,False,4644,CTSpawn,hypex,76561198254686734,glock,17293822569105850372,47842867554,,0
2,False,CTSpawn,slaxz-,76561198064353169,TopofMid,Lake,76561198337484196,False,False,27.982517,...,False,4723,Middle,Snax,76561197982141573,glock,17293822569183969284,35627379237,,0
3,False,None,None,None,TopofMid,Lake,76561198337484196,False,False,34.145321,...,False,4825,Catwalk,REZ,76561198034172415,glock,17293822569183969284,35627379237,,0
4,False,None,None,None,CTSpawn,Tauson,76561198316706174,False,False,11.148964,...,False,5043,CTSpawn,JBa,76561198184153575,usp_silencer,17293822569182462013,47237139810,,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,False,None,None,None,TRamp,Tauson,76561198316706174,False,False,23.483894,...,False,212052,Stairs,JBa,76561198184153575,ak47,17293822569135865863,33256536744,,0
170,False,PalaceInterior,ztr,76561199009468655,PalaceInterior,REZ,76561198034172415,False,False,19.454647,...,False,212166,Scaffolding,slaxz-,76561198064353169,ak47,17293822569144582151,41459218028,,0
171,False,None,None,None,BombsiteA,REZ,76561198034172415,False,False,21.123171,...,False,212385,BombsiteA,Lake,76561198337484196,ak47,17293822569144582151,41459218028,,0
172,False,BombsiteA,hypex,76561198254686734,Connector,Snax,76561197982141573,False,False,12.341070,...,False,212681,BombsiteA,Swisher,76561198183003671,ak47,17293822569179447303,48224722730,,0


In [56]:
players['start_tick']

0            1358
1            1358
2            1358
3            1358
4            1358
            ...  
2124165    210936
2124166    210936
2124167    210936
2124168    210936
2124169    210936
Name: start_tick, Length: 1308980, dtype: int32

In [1]:
test['rounds']

NameError: name 'test' is not defined

In [ ]:
parser = DemoParser(files[0])

In [ ]:
parser.

In [45]:
test = parser.parse_ticks(['inventory'])

In [52]:
inv = test['inventory'].explode()
uni = inv.unique()
print(uni)

['Skeleton Knife' 'Glock-18' 'Butterfly Knife' 'C4 Explosive' 'M9 Bayonet'
 'USP-S' 'Talon Knife' 'Karambit' 'Smoke Grenade' 'Flashbang' 'Molotov'
 nan 'AK-47' 'Five-SeveN' 'Galil AR' 'MAC-10' 'MP9'
 'High Explosive Grenade' 'Desert Eagle' 'AWP' 'M4A1-S'
 'Incendiary Grenade' 'Zeus x27' 'Tec-9' 'Dual Berettas' 'Decoy Grenade'
 'FAMAS' 'SSG 08' 'Bayonet' 'M4A4' 'P250']


In [ ]:
inv[inv=='Zeus x27']

'Galil AR'

In [28]:
len(players['ticks']['steamid'].unique())

10

In [59]:
test.iloc[437154]

inventory    [M9 Bayonet, USP-S, AWP, Smoke Grenade, Incend...
tick                                                     43722
steamid                                      76561198254686734
name                                                     hypex
Name: 437154, dtype: object

In [6]:
parser.parse_event('bomb_planted')

,site,tick,user_name,user_steamid
0,781,3815,Swisher,76561198183003671
1,781,14286,Lake,76561198337484196
2,1012,23645,Lake,76561198337484196
3,781,33132,Swisher,76561198183003671
4,1012,67236,slaxz-,76561198064353169
5,781,77463,Lake,76561198337484196
6,781,83421,Swisher,76561198183003671
7,781,90255,s1n,76561198190463461
8,1012,96831,Snax,76561197982141573
9,781,104924,hypex,76561198254686734


In [7]:
parser.parse_event('bomb_exploded')

,site,tick,user_name,user_steamid
0,781,6439,Swisher,76561198183003671
1,1012,26269,Lake,76561198337484196
2,781,86045,Swisher,76561198183003671
3,1012,99455,Snax,76561197982141573
4,781,107548,hypex,76561198254686734
5,1012,114117,hypex,76561198254686734
6,1012,143776,ztr,76561199009468655
7,781,207113,Tauson,76561198316706174


In [8]:
parser.parse_event('bomb_defused')

,site,tick,user_name,user_steamid
0,781,35431,Tauson,76561198316706174
1,781,79280,Snax,76561197982141573
2,1012,133386,JBa,76561198184153575
